<a href="https://colab.research.google.com/github/yanchenliu-cxk/ESM/blob/main/EC_and_usual.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# =========================================================
# Cell 6A：断线后重新部署 EC→Cofactor 映射所需环境
# 不需要重跑 CLEAN-Contact / Cell 1-5
# =========================================================

from google.colab import drive
drive.mount('/content/drive')

!pip install -q --upgrade pandas openpyxl requests tqdm tenacity

import os
import re
import json
import time
import hashlib
from io import StringIO
from pathlib import Path
from collections import defaultdict

import pandas as pd
import requests
from requests.adapters import HTTPAdapter, Retry
from tqdm.auto import tqdm

print("✅ Cell 6 环境恢复完成。")
print("pandas:", pd.__version__)
print("requests:", requests.__version__)

# 检查 Cell 5 输出文件
INPUT_XLSX = Path("/content/drive/MyDrive/Horizyn_Checkpoints/CLEANContact_my_enzymes_EC_predictions.xlsx")

if not INPUT_XLSX.exists():
    print("⚠️ 没有在默认路径找到 Cell 5 结果，正在搜索可能的 EC 预测表...")
    candidates = list(Path("/content/drive/MyDrive").rglob("*CLEANContact*EC*prediction*.xlsx"))
    for i, p in enumerate(candidates[:20], 1):
        print(f"{i}. {p}")
    raise FileNotFoundError("请确认 Cell 5 的 EC 预测结果文件路径。")
else:
    print(f"✅ 找到 Cell 5 EC 预测表：{INPUT_XLSX}")

Mounted at /content/drive
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 119.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 7.7 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 3.0.3 which is incompatible.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.
cudf-cu12 26.2.1 requires pandas<2.4.0,>=2.0, but you have pandas 3.0.3 which is incompatible.
gradio 5.50.0 requires pandas<3.0,>=1.0, but you have pandas 3.0.3 which is incompatible.
db-dtypes 1.5.1 requires pandas<3.0.0,>=1.5.3, but you have pandas 3.0.3 which is incompatible.
bqplot 0.12.45 requires pandas<3.0.0,>=1.0.0, but you have pandas 3.0.3 which is incompati

In [2]:
# =========================================================
# Cell 6B：EC → Cofactor 多证据映射
# Sources: KEGG + UniProt + Rhea + M-CSA + EC-class prior
# 修正版：已修复 for_col_search 缩进错误
# =========================================================

import os
import re
import json
import time
import hashlib
from io import StringIO
from pathlib import Path
from collections import defaultdict

import pandas as pd
import requests
from requests.adapters import HTTPAdapter, Retry
from tqdm.auto import tqdm

# -----------------------------
# 0. 路径设置
# -----------------------------
INPUT_XLSX = Path("/content/drive/MyDrive/Horizyn_Checkpoints/CLEANContact_my_enzymes_EC_predictions.xlsx")
OUT_DIR = Path("/content/drive/MyDrive/Horizyn_Checkpoints/cofactor_mapping")
CACHE_DIR = Path("/content/ec_cofactor_cache")

OUT_DIR.mkdir(parents=True, exist_ok=True)
CACHE_DIR.mkdir(parents=True, exist_ok=True)

assert INPUT_XLSX.exists(), f"找不到 EC 预测表: {INPUT_XLSX}"

# -----------------------------
# 1. requests session with retry
# -----------------------------
def make_session():
    session = requests.Session()
    retries = Retry(
        total=5,
        backoff_factor=1.0,
        status_forcelist=[429, 500, 502, 503, 504],
        allowed_methods=["GET", "POST"]
    )
    session.mount("https://", HTTPAdapter(max_retries=retries))
    session.mount("http://", HTTPAdapter(max_retries=retries))
    session.headers.update({"User-Agent": "EC-Cofactor-Mapping-Colab/1.0"})
    return session

SESSION = make_session()

def cached_get(url, params=None, suffix="txt", sleep=0.15):
    """
    带缓存的 GET，避免重复请求数据库。
    """
    key = url + "?" + json.dumps(params or {}, sort_keys=True)
    fname = CACHE_DIR / (hashlib.md5(key.encode()).hexdigest() + f".{suffix}")

    if fname.exists() and fname.stat().st_size > 0:
        return fname.read_text(encoding="utf-8", errors="ignore")

    r = SESSION.get(url, params=params, timeout=40)
    r.raise_for_status()
    text = r.text

    fname.write_text(text, encoding="utf-8")
    time.sleep(sleep)
    return text

# -----------------------------
# 2. 读取 CLEAN-Contact EC 表
# -----------------------------
df_pred = pd.read_excel(INPUT_XLSX)

print("输入表列名：")
print(list(df_pred.columns))

candidate_ec_cols = [
    "Predicted_EC_4digit",
    "Predicted_EC_4_digit",
    "Top1_EC",
    "EC",
    "Predicted_EC"
]

ec_col = None

# 这里是你刚才报错的地方：已经修复，没有多余空格
for_col_search = [c for c in candidate_ec_cols if c in df_pred.columns]

if for_col_search:
    ec_col = for_col_search[0]
else:
    for c in df_pred.columns:
        if "EC" in c.upper():
            ec_col = c
            break

assert ec_col is not None, "没有找到 EC 列。请检查 Cell 5 输出表。"

id_col = None
for c in ["Enzyme_ID", "Entry", "CleanContact_Entry", "Original_ID"]:
    if c in df_pred.columns:
        id_col = c
        break

if id_col is None:
    id_col = df_pred.columns[0]

print(f"✅ 使用 ID 列: {id_col}")
print(f"✅ 使用 EC 列: {ec_col}")

def parse_ecs(x):
    """
    兼容：
    1.1.1.1
    1.1.1.1;2.7.1.1
    EC:1.1.1.1/0.42
    """
    if pd.isna(x):
        return []
    s = str(x)
    ecs = re.findall(r"(?:EC:)?([0-9-]+\.[0-9-]+\.[0-9-]+\.[0-9-]+)", s)
    out = []
    for e in ecs:
        if e not in out:
            out.append(e)
    return out

df_pred["Parsed_ECs"] = df_pred[ec_col].apply(parse_ecs)
all_ecs = sorted(set(e for ecs in df_pred["Parsed_ECs"] for e in ecs))

print(f"总 enzyme 数: {len(df_pred)}")
print(f"唯一 EC 数: {len(all_ecs)}")
print("前 10 个 EC:", all_ecs[:10])

if len(all_ecs) == 0:
    raise ValueError("没有解析到任何 EC。请检查 EC 输出格式。")

# -----------------------------
# 3. 辅因子词典
# -----------------------------
COFACTOR_PATTERNS = {
    "NAD+": [
        r"\bNAD\+\b", r"\bNADH\b", r"\bNAD\b(?!P)", r"nicotinamide adenine dinucleotide"
    ],
    "NADP+": [
        r"\bNADP\+?\b", r"\bNADPH\b", r"nicotinamide adenine dinucleotide phosphate"
    ],
    "FAD": [
        r"\bFAD\b", r"flavin adenine dinucleotide"
    ],
    "FMN": [
        r"\bFMN\b", r"flavin mononucleotide"
    ],
    "PLP": [
        r"\bPLP\b", r"pyridoxal(?: |-)?5'?[- ]?phosphate", r"pyridoxal phosphate"
    ],
    "PQQ": [
        r"\bPQQ\b", r"pyrroloquinoline quinone"
    ],
    "SAM": [
        r"\bSAM\b", r"S-adenosyl(?:-L)?-?methionine", r"adenosylmethionine"
    ],
    "CoA": [
        r"\bCoA\b", r"coenzyme A"
    ],
    "Biotin": [
        r"\bbiotin\b", r"biocytin"
    ],
    "TPP": [
        r"\bTPP\b", r"thiamine diphosphate", r"thiamin diphosphate", r"thiamine pyrophosphate"
    ],
    "Heme": [
        r"\bheme\b", r"\bhaem\b", r"\bHEM\b", r"protoheme", r"porphyrin"
    ],
    "Cobalamin/B12": [
        r"cobalamin", r"vitamin B12", r"\bB12\b", r"adenosylcobalamin", r"methylcobalamin"
    ],
    "Tetrahydrofolate": [
        r"tetrahydrofolate", r"\bTHF\b", r"folate"
    ],
    "Lipoate": [
        r"lipoate", r"lipoamide", r"lipoic acid"
    ],
    "Molybdopterin": [
        r"molybdopterin", r"molybdenum cofactor", r"\bMoco\b"
    ],
    "Fe-S cluster": [
        r"iron[- ]sulfur", r"\bFe-S\b", r"2Fe-2S", r"4Fe-4S", r"\[2Fe-2S\]", r"\[4Fe-4S\]"
    ],
    "Metal:Mg": [
        r"magnesium", r"\bMg2\+\b", r"\bMg\(2\+\)", r"\bMg\+\+"
    ],
    "Metal:Zn": [
        r"\bzinc\b", r"\bZn2\+\b", r"\bZn\(2\+\)", r"\bZn\+\+"
    ],
    "Metal:Fe": [
        r"\biron\b", r"\bFe2\+\b", r"\bFe3\+\b", r"\bFe\(2\+\)", r"\bFe\(3\+\)"
    ],
    "Metal:Mn": [
        r"manganese", r"\bMn2\+\b", r"\bMn\(2\+\)"
    ],
    "Metal:Cu": [
        r"\bcopper\b", r"\bCu2\+\b", r"\bCu\(2\+\)"
    ],
    "Metal:Ca": [
        r"\bcalcium\b", r"\bCa2\+\b", r"\bCa\(2\+\)"
    ],
    "Metal:Ni": [
        r"\bnickel\b", r"\bNi2\+\b", r"\bNi\(2\+\)"
    ],
    "Metal:Co": [
        r"\bcobalt\b", r"\bCo2\+\b", r"\bCo\(2\+\)"
    ],
    "Quinone": [
        r"ubiquinone", r"menaquinone", r"\bquinone\b"
    ],
}

COFACTOR_MODELING_HINTS = {
    "NAD+": "PDB CCD: NAD；NADH/NAD+ 需根据反应方向选择氧化/还原态",
    "NADP+": "PDB CCD: NAP；NADPH/NADP+ 需根据反应方向选择氧化/还原态",
    "FAD": "PDB CCD: FAD",
    "FMN": "PDB CCD: FMN",
    "PLP": "PDB CCD: PLP；注意可能与 Lys 形成 Schiff base",
    "PQQ": "PDB CCD: PQQ",
    "SAM": "PDB CCD: SAM；产物可为 SAH",
    "CoA": "PDB CCD: COA；酰基-CoA 需根据底物改写",
    "Biotin": "PDB CCD: BTN；注意 biotinyl-lysine",
    "TPP": "PDB CCD: TPP",
    "Heme": "PDB CCD: HEM/HEC/HEA，需按酶家族确认 heme 类型",
    "Cobalamin/B12": "AdoCbl/MeCbl 类型需根据反应确认",
    "Tetrahydrofolate": "THF/5,10-methylene-THF 等需根据反应确认",
    "Lipoate": "通常共价连接到 Lys，需要按 lipoamide arm 建模",
    "Molybdopterin": "MPT/MGD 等需按家族确认",
    "Fe-S cluster": "PDB CCD: SF4/F3S/FES，需按 Cys motif 确认",
    "Metal:Mg": "Ion: MG",
    "Metal:Zn": "Ion: ZN",
    "Metal:Fe": "Ion: FE/FE2/FE3，需按价态确认",
    "Metal:Mn": "Ion: MN",
    "Metal:Cu": "Ion: CU/CU1",
    "Metal:Ca": "Ion: CA",
    "Metal:Ni": "Ion: NI",
    "Metal:Co": "Ion: CO",
    "Quinone": "UQ/MQ/quinone 类需根据反应确认",
}

def detect_cofactors(text):
    if not text:
        return set()
    hits = set()
    for name, patterns in COFACTOR_PATTERNS.items():
        for pat in patterns:
            if re.search(pat, str(text), flags=re.IGNORECASE):
                hits.add(name)
                break
    return hits

# -----------------------------
# 4. EC-class weak prior
# -----------------------------
def ec_class_prior(ec):
    priors = set()
    explanation = []

    parts = str(ec).split(".")
    if len(parts) != 4:
        return priors, ""

    c1, c2, c3, c4 = parts

    if c1 == "1":
        priors.update(["NAD+", "NADP+", "FAD", "FMN", "Fe-S cluster", "Metal:Fe", "Heme"])
        explanation.append("EC 1 oxidoreductase: redox enzymes often use NAD(P), flavins, heme, Fe-S or metals")

    if c1 == "2":
        explanation.append("EC 2 transferase: cofactor depends on transferred group")
        if c2 == "1":
            priors.update(["SAM", "Tetrahydrofolate"])
        if c2 == "3":
            priors.add("CoA")
        if c2 == "6":
            priors.add("PLP")

    if c1 == "4":
        priors.update(["PLP", "TPP", "Metal:Mg", "Metal:Mn"])
        explanation.append("EC 4 lyase: some subclasses use PLP, TPP or divalent metals")

    if c1 == "5":
        priors.update(["Cobalamin/B12", "Metal:Mg", "Metal:Zn"])
        explanation.append("EC 5 isomerase: some subclasses use B12 or metals")

    if c1 == "6":
        priors.add("Metal:Mg")
        explanation.append("EC 6 ligase: Mg2+ is frequently required with ATP/NTP")

    return priors, "; ".join(explanation)

# -----------------------------
# 5. KEGG EC 查询
# -----------------------------
def parse_kegg_fields(text):
    fields = defaultdict(list)
    current = None

    for line in text.splitlines():
        if not line.strip():
            continue

        key = line[:12].strip()
        val = line[12:].strip()

        if key:
            current = key
            fields[current].append(val)
        elif current:
            fields[current].append(val)

    return {k: " ".join(v) for k, v in fields.items()}

def query_kegg_ec(ec):
    url = f"https://rest.kegg.jp/get/ec:{ec}"

    try:
        text = cached_get(url, suffix="kegg.txt")

        if "Entry not found" in text or len(text.strip()) == 0:
            return {
                "ok": False,
                "hits": set(),
                "cofactor_text": "",
                "reaction_text": "",
                "raw": ""
            }

        fields = parse_kegg_fields(text)

        cofactor_text = fields.get("COFACTOR", "")
        reaction_text = " | ".join([
            fields.get("REACTION", ""),
            fields.get("SUBSTRATE", ""),
            fields.get("PRODUCT", ""),
            fields.get("COMMENT", ""),
            fields.get("EFFECTOR", ""),
            fields.get("CLASS", ""),
            fields.get("SYSNAME", ""),
        ])

        hits = detect_cofactors(cofactor_text + " " + reaction_text)

        return {
            "ok": True,
            "hits": hits,
            "cofactor_text": cofactor_text,
            "reaction_text": reaction_text,
            "raw": text[:5000]
        }

    except Exception as e:
        return {
            "ok": False,
            "error": str(e),
            "hits": set(),
            "cofactor_text": "",
            "reaction_text": "",
            "raw": ""
        }

# -----------------------------
# 6. UniProt reviewed EC 查询
# -----------------------------
def query_uniprot_ec(ec, size=25):
    url = "https://rest.uniprot.org/uniprotkb/search"
    params = {
        "query": f"(ec:{ec}) AND (reviewed:true)",
        "format": "tsv",
        "fields": "accession,id,protein_name,organism_name,ec,cc_cofactor,cc_catalytic_activity",
        "size": str(size),
    }

    try:
        text = cached_get(url, params=params, suffix="uniprot.tsv")

        if not text.strip():
            return {
                "ok": True,
                "hits": set(),
                "cofactor_text": "",
                "catalytic_activity_text": "",
                "n_entries": 0,
                "raw": ""
            }

        df_u = pd.read_csv(StringIO(text), sep="\t")

        if df_u.empty:
            return {
                "ok": True,
                "hits": set(),
                "cofactor_text": "",
                "catalytic_activity_text": "",
                "n_entries": 0,
                "raw": text[:3000]
            }

        cofactor_cols = [c for c in df_u.columns if "cofactor" in c.lower()]
        catalytic_cols = [c for c in df_u.columns if "catalytic" in c.lower()]

        cofactor_texts = []
        for c in cofactor_cols:
            cofactor_texts.extend(df_u[c].dropna().astype(str).tolist())
        cofactor_text = " | ".join(cofactor_texts)

        catalytic_texts = []
        for c in catalytic_cols:
            catalytic_texts.extend(df_u[c].dropna().astype(str).tolist())
        catalytic_text = " | ".join(catalytic_texts)

        hits = detect_cofactors(cofactor_text + " " + catalytic_text)

        return {
            "ok": True,
            "hits": hits,
            "cofactor_text": cofactor_text[:4000],
            "catalytic_activity_text": catalytic_text[:4000],
            "n_entries": len(df_u),
            "raw": text[:5000]
        }

    except Exception as e:
        return {
            "ok": False,
            "error": str(e),
            "hits": set(),
            "cofactor_text": "",
            "catalytic_activity_text": "",
            "n_entries": 0,
            "raw": ""
        }

# -----------------------------
# 7. Rhea EC → Rhea ID
# -----------------------------
def load_rhea2ec():
    url = "https://ftp.expasy.org/databases/rhea/tsv/rhea2ec.tsv"
    text = cached_get(url, suffix="rhea2ec.tsv")

    df_r = pd.read_csv(StringIO(text), sep="\t", dtype=str, comment="#")
    cols = list(df_r.columns)

    ec_col_r = None
    for c in cols:
        if "ec" in c.lower():
            ec_col_r = c
            break
    if ec_col_r is None:
        ec_col_r = cols[-1]

    rhea_col = None
    for c in cols:
        if "rhea" in c.lower():
            rhea_col = c
            break
    if rhea_col is None:
        rhea_col = cols[0]

    df_r = df_r[[rhea_col, ec_col_r]].rename(columns={rhea_col: "RHEA_ID", ec_col_r: "EC"})
    df_r["EC"] = df_r["EC"].astype(str).str.strip()
    df_r["RHEA_ID"] = df_r["RHEA_ID"].astype(str).str.replace("RHEA:", "", regex=False).str.strip()

    return df_r.dropna()

try:
    rhea2ec_df = load_rhea2ec()
    print(f"✅ Rhea2EC 映射加载完成: {len(rhea2ec_df)} rows")
except Exception as e:
    print(f"⚠️ Rhea2EC 加载失败: {e}")
    rhea2ec_df = pd.DataFrame(columns=["RHEA_ID", "EC"])

def query_rhea_ids_by_ec(ec):
    if rhea2ec_df.empty:
        return []
    sub = rhea2ec_df[rhea2ec_df["EC"] == ec]
    return sorted(set(sub["RHEA_ID"].dropna().astype(str).tolist()))

# -----------------------------
# 8. M-CSA API 查询
# -----------------------------
def flatten_json_text(obj, max_chars=8000):
    chunks = []

    def rec(x):
        if isinstance(x, dict):
            for k, v in x.items():
                chunks.append(str(k))
                rec(v)
        elif isinstance(x, list):
            for v in x:
                rec(v)
        else:
            if x is not None:
                chunks.append(str(x))

    rec(obj)
    return " ".join(chunks)[:max_chars]

def query_mcsa_ec(ec):
    url = "https://www.ebi.ac.uk/thornton-srv/m-csa/api/entries/"
    params = {
        "format": "json",
        "entries.reactions.ecs.codes": ec
    }

    try:
        text = cached_get(url, params=params, suffix="mcsa.json", sleep=0.25)
        data = json.loads(text)

        flat = flatten_json_text(data)
        hits = detect_cofactors(flat)

        mcsa_ids = sorted(set(re.findall(r'"?mcsa_id"?\s*[:=]\s*"?([0-9]+)"?', text)))

        return {
            "ok": True,
            "hits": hits,
            "mcsa_ids": mcsa_ids,
            "summary_text": flat[:4000],
            "raw": text[:5000]
        }

    except Exception as e:
        return {
            "ok": False,
            "error": str(e),
            "hits": set(),
            "mcsa_ids": [],
            "summary_text": "",
            "raw": ""
        }

# -----------------------------
# 9. 单个 EC 汇总评分
# -----------------------------
def score_ec_cofactors(ec):
    kegg = query_kegg_ec(ec)
    uniprot = query_uniprot_ec(ec, size=25)
    mcsa = query_mcsa_ec(ec)
    rhea_ids = query_rhea_ids_by_ec(ec)
    prior_hits, prior_note = ec_class_prior(ec)

    evidence = defaultdict(lambda: {
        "score": 0.0,
        "sources": [],
        "details": []
    })

    # M-CSA：机制证据
    for c in mcsa.get("hits", set()):
        evidence[c]["score"] += 0.40
        evidence[c]["sources"].append("M-CSA")
        evidence[c]["details"].append("M-CSA mechanism/API text contains cofactor keyword")

    # UniProt reviewed entries
    for c in uniprot.get("hits", set()):
        evidence[c]["score"] += 0.30
        evidence[c]["sources"].append("UniProt")
        evidence[c]["details"].append("UniProt reviewed entries mention cofactor/catalytic activity")

    # KEGG
    kegg_hits = kegg.get("hits", set())
    kegg_cofactor_hits = detect_cofactors(kegg.get("cofactor_text", ""))

    for c in kegg_hits:
        if c in kegg_cofactor_hits:
            evidence[c]["score"] += 0.25
            evidence[c]["details"].append("KEGG COFACTOR field")
        else:
            evidence[c]["score"] += 0.15
            evidence[c]["details"].append("KEGG reaction/comment keyword")

        evidence[c]["sources"].append("KEGG")

    # Rhea：反应映射支持
    if rhea_ids:
        for c in set(kegg_hits) | set(uniprot.get("hits", set())):
            evidence[c]["score"] += 0.05
            evidence[c]["sources"].append("Rhea")
            evidence[c]["details"].append(f"EC maps to Rhea IDs: {','.join(rhea_ids[:5])}")

    # EC class prior：弱先验
    for c in prior_hits:
        evidence[c]["score"] += 0.05
        evidence[c]["sources"].append("EC-class-prior")
        evidence[c]["details"].append(prior_note)

    rows = []

    for c, ev in evidence.items():
        score = min(ev["score"], 1.0)

        if score >= 0.70:
            conf = "High"
        elif score >= 0.40:
            conf = "Probable"
        elif score >= 0.20:
            conf = "Possible"
        else:
            conf = "Weak"

        rows.append({
            "EC": ec,
            "Cofactor": c,
            "Cofactor_Evidence_Score": round(score, 3),
            "Confidence": conf,
            "Evidence_Sources": ";".join(sorted(set(ev["sources"]))),
            "Evidence_Details": " | ".join(sorted(set(ev["details"]))),
            "Modeling_Hint": COFACTOR_MODELING_HINTS.get(c, ""),
            "Rhea_IDs": ";".join(rhea_ids),
            "MCSA_IDs": ";".join(mcsa.get("mcsa_ids", [])),
            "KEGG_Cofactor_Text": kegg.get("cofactor_text", ""),
            "UniProt_Cofactor_Text": uniprot.get("cofactor_text", ""),
            "UniProt_Catalytic_Activity_Text": uniprot.get("catalytic_activity_text", ""),
            "EC_Class_Prior_Note": prior_note,
            "KEGG_OK": kegg.get("ok", False),
            "UniProt_Reviewed_Entries": uniprot.get("n_entries", 0),
            "MCSA_OK": mcsa.get("ok", False),
        })

    if not rows:
        rows.append({
            "EC": ec,
            "Cofactor": "",
            "Cofactor_Evidence_Score": 0.0,
            "Confidence": "No_public_evidence",
            "Evidence_Sources": "",
            "Evidence_Details": "No cofactor keyword detected from KEGG/UniProt/M-CSA; check BRENDA/manual literature.",
            "Modeling_Hint": "",
            "Rhea_IDs": ";".join(rhea_ids),
            "MCSA_IDs": ";".join(mcsa.get("mcsa_ids", [])),
            "KEGG_Cofactor_Text": kegg.get("cofactor_text", ""),
            "UniProt_Cofactor_Text": uniprot.get("cofactor_text", ""),
            "UniProt_Catalytic_Activity_Text": uniprot.get("catalytic_activity_text", ""),
            "EC_Class_Prior_Note": prior_note,
            "KEGG_OK": kegg.get("ok", False),
            "UniProt_Reviewed_Entries": uniprot.get("n_entries", 0),
            "MCSA_OK": mcsa.get("ok", False),
        })

    return rows

# -----------------------------
# 10. 批量运行 EC 映射
# -----------------------------
all_ec_evidence_rows = []

for ec in tqdm(all_ecs, desc="Mapping EC to cofactors"):
    try:
        rows = score_ec_cofactors(ec)
        all_ec_evidence_rows.extend(rows)
    except Exception as e:
        print(f"⚠️ EC {ec} failed: {e}")
        all_ec_evidence_rows.append({
            "EC": ec,
            "Cofactor": "",
            "Cofactor_Evidence_Score": 0.0,
            "Confidence": "Query_failed",
            "Evidence_Sources": "",
            "Evidence_Details": str(e),
        })

df_ec_evidence = pd.DataFrame(all_ec_evidence_rows)

# -----------------------------
# 11. 回填到 enzyme 层面
# -----------------------------
def summarize_for_enzyme(ec_list):
    candidates = []

    for ec in ec_list:
        sub = df_ec_evidence[df_ec_evidence["EC"] == ec].copy()

        if sub.empty:
            continue

        sub = sub[sub["Cofactor"].astype(str).str.len() > 0]

        for _, r in sub.iterrows():
            candidates.append(r.to_dict())

    if not candidates:
        return pd.Series({
            "Cofactor_Category": "No_public_evidence",
            "Top_Cofactors": "",
            "Top_Cofactor_Score": 0.0,
            "Top_Cofactor_Confidence": "No_public_evidence",
            "All_Cofactor_Candidates": "",
            "Cofactor_Evidence_Sources": "",
            "Modeling_Hints": "",
        })

    cand_df = pd.DataFrame(candidates)
    cand_df = cand_df.sort_values("Cofactor_Evidence_Score", ascending=False)

    top_score = float(cand_df.iloc[0]["Cofactor_Evidence_Score"])

    if top_score >= 0.70:
        category = "High-confidence cofactor-dependent"
    elif top_score >= 0.40:
        category = "Probable cofactor-dependent"
    elif top_score >= 0.20:
        category = "Possible cofactor-dependent"
    else:
        category = "Weak cofactor evidence"

    agg = (
        cand_df.groupby("Cofactor", as_index=False)
        .agg({
            "Cofactor_Evidence_Score": "max",
            "Confidence": lambda x: ";".join(sorted(set(map(str, x)))),
            "Evidence_Sources": lambda x: ";".join(sorted(set(";".join(map(str, x)).split(";")))),
            "Modeling_Hint": lambda x: " | ".join(sorted(set([str(v) for v in x if str(v) not in ["nan", "None", ""]]))),
        })
        .sort_values("Cofactor_Evidence_Score", ascending=False)
    )

    top_cofactors = ";".join(
        f"{r.Cofactor}({r.Cofactor_Evidence_Score:.2f})"
        for _, r in agg.head(5).iterrows()
    )

    all_candidates = ";".join(
        f"{r.Cofactor}:{r.Cofactor_Evidence_Score:.2f}"
        for _, r in agg.iterrows()
    )

    sources = ";".join(sorted(set(";".join(agg["Evidence_Sources"].astype(str)).split(";"))))

    hints = " || ".join(
        f"{r.Cofactor}: {r.Modeling_Hint}"
        for _, r in agg.head(5).iterrows()
        if str(r.Modeling_Hint).strip()
    )

    return pd.Series({
        "Cofactor_Category": category,
        "Top_Cofactors": top_cofactors,
        "Top_Cofactor_Score": round(top_score, 3),
        "Top_Cofactor_Confidence": cand_df.iloc[0]["Confidence"],
        "All_Cofactor_Candidates": all_candidates,
        "Cofactor_Evidence_Sources": sources,
        "Modeling_Hints": hints,
    })

df_enzyme = df_pred.copy()
summary_cols = df_enzyme["Parsed_ECs"].apply(summarize_for_enzyme)
df_enzyme = pd.concat([df_enzyme, summary_cols], axis=1)

# -----------------------------
# 12. 输出 Excel
# -----------------------------
out_xlsx = OUT_DIR / "EC_to_Cofactor_MultiEvidence_Mapping.xlsx"
out_csv = OUT_DIR / "Enzyme_level_Cofactor_Mapping.csv"

with pd.ExcelWriter(out_xlsx, engine="openpyxl") as writer:
    df_enzyme.to_excel(writer, index=False, sheet_name="enzyme_level")
    df_ec_evidence.to_excel(writer, index=False, sheet_name="ec_level_evidence")

    vocab_rows = []
    for k, patterns in COFACTOR_PATTERNS.items():
        vocab_rows.append({
            "Cofactor": k,
            "Regex_Patterns": " | ".join(patterns),
            "Modeling_Hint": COFACTOR_MODELING_HINTS.get(k, "")
        })

    pd.DataFrame(vocab_rows).to_excel(writer, index=False, sheet_name="cofactor_vocab")

df_enzyme.to_csv(out_csv, index=False)

print("\n✅ Cofactor 多证据映射完成")
print(f"Excel: {out_xlsx}")
print(f"CSV:   {out_csv}")

print("\n分类统计：")
print(df_enzyme["Cofactor_Category"].value_counts(dropna=False))

display(
    df_enzyme[
        [id_col, ec_col, "Cofactor_Category", "Top_Cofactors", "Cofactor_Evidence_Sources"]
    ].head(20)
)

输入表列名：
['Enzyme_ID', 'Predicted_EC_4digit', 'CLEANContact_Distance', 'Top1_EC', 'Top1_Distance', 'Num_Predicted_ECs', 'Original_ID', 'CleanContact_Entry', 'Sequence_length']
✅ 使用 ID 列: Enzyme_ID
✅ 使用 EC 列: Predicted_EC_4digit
总 enzyme 数: 500
唯一 EC 数: 427
前 10 个 EC: ['1.1.1.120', '1.1.1.122', '1.1.1.133', '1.1.1.145', '1.1.1.146', '1.1.1.17', '1.1.1.190', '1.1.1.191', '1.1.1.201', '1.1.1.210']
✅ Rhea2EC 映射加载完成: 7902 rows


Mapping EC to cofactors:   0%|          | 0/427 [00:00<?, ?it/s]


✅ Cofactor 多证据映射完成
Excel: /content/drive/MyDrive/Horizyn_Checkpoints/cofactor_mapping/EC_to_Cofactor_MultiEvidence_Mapping.xlsx
CSV:   /content/drive/MyDrive/Horizyn_Checkpoints/cofactor_mapping/Enzyme_level_Cofactor_Mapping.csv

分类统计：
Cofactor_Category
Probable cofactor-dependent           156
Possible cofactor-dependent           139
No_public_evidence                     85
High-confidence cofactor-dependent     67
Weak cofactor evidence                 53
Name: count, dtype: int64


,Enzyme_ID,Predicted_EC_4digit,Cofactor_Category,Top_Cofactors,Cofactor_Evidence_Sources
0,NODE_1_length_436095_cov_65.793628_39,2.7.1.113;2.7.1.76,Probable cofactor-dependent,Metal:Mg(0.40),M-CSA
1,NODE_1_length_436095_cov_65.793628_41,2.7.1.113,Probable cofactor-dependent,Metal:Mg(0.40),M-CSA
2,NODE_1_length_436095_cov_65.793628_87,1.1.1.355,Probable cofactor-dependent,NADP+(0.50);NAD+(0.20);FAD(0.05);Fe-S cluster(...,EC-class-prior;KEGG;UniProt
3,NODE_1_length_436095_cov_65.793628_95,2.3.1.30,Probable cofactor-dependent,CoA(0.50),EC-class-prior;KEGG;UniProt
4,NODE_1_length_436095_cov_65.793628_115,3.1.6.1,Possible cofactor-dependent,Metal:Ca(0.30);Heme(0.15);Metal:Fe(0.15);Metal...,KEGG;UniProt
5,NODE_1_length_436095_cov_65.793628_119,3.1.21.2,Possible cofactor-dependent,Metal:Mn(0.30);Metal:Zn(0.30),UniProt
6,NODE_1_length_436095_cov_65.793628_142,5.4.4.1;1.17.4.4,Weak cofactor evidence,Quinone(0.15);FAD(0.05);Cobalamin/B12(0.05);FM...,EC-class-prior;KEGG
7,NODE_1_length_436095_cov_65.793628_144,3.1.1.5,High-confidence cofactor-dependent,CoA(0.70);Metal:Ca(0.30),M-CSA;UniProt
8,NODE_1_length_436095_cov_65.793628_198,3.1.4.46,Possible cofactor-dependent,Metal:Ca(0.30);Metal:Co(0.30);Metal:Fe(0.30);M...,UniProt
9,NODE_1_length_436095_cov_65.793628_199,5.4.4.1,Weak cofactor evidence,Cobalamin/B12(0.05);Metal:Mg(0.05);Metal:Zn(0.05),EC-class-prior


In [3]:
import pandas as pd
from pathlib import Path

mapping_xlsx = Path("/content/drive/MyDrive/Horizyn_Checkpoints/cofactor_mapping/EC_to_Cofactor_MultiEvidence_Mapping.xlsx")
out_dir = Path("/content/drive/MyDrive/Horizyn_Checkpoints/cofactor_mapping")
out_dir.mkdir(parents=True, exist_ok=True)

df = pd.read_excel(mapping_xlsx, sheet_name="enzyme_level")

# 1. 主建模集：High + Probable
df_primary = df[
    df["Cofactor_Category"].isin([
        "High-confidence cofactor-dependent",
        "Probable cofactor-dependent"
    ])
].copy()

# 2. 二级验证集：Possible
df_possible = df[
    df["Cofactor_Category"].eq("Possible cofactor-dependent")
].copy()

# 3. 暂缓集：Weak + No_public_evidence
df_low = df[
    df["Cofactor_Category"].isin([
        "Weak cofactor evidence",
        "No_public_evidence"
    ])
].copy()

df_primary.to_excel(out_dir / "01_Primary_High_Probable_Cofactor_Holo_Modeling.xlsx", index=False)
df_possible.to_excel(out_dir / "02_Secondary_Possible_Cofactor_Check.xlsx", index=False)
df_low.to_excel(out_dir / "03_Low_or_No_Cofactor_Evidence.xlsx", index=False)

print("主建模集 High + Probable:", len(df_primary))
print("二级验证集 Possible:", len(df_possible))
print("暂缓集 Weak + No evidence:", len(df_low))

display(df_primary[[
    "Enzyme_ID",
    "Predicted_EC_4digit",
    "Top_Cofactors",
    "Top_Cofactor_Score",
    "Top_Cofactor_Confidence",
    "Cofactor_Evidence_Sources",
    "Modeling_Hints"
]].head(20))

主建模集 High + Probable: 223
二级验证集 Possible: 139
暂缓集 Weak + No evidence: 138


,Enzyme_ID,Predicted_EC_4digit,Top_Cofactors,Top_Cofactor_Score,Top_Cofactor_Confidence,Cofactor_Evidence_Sources,Modeling_Hints
0,NODE_1_length_436095_cov_65.793628_39,2.7.1.113;2.7.1.76,Metal:Mg(0.40),0.40,Probable,M-CSA,Metal:Mg: Ion: MG
1,NODE_1_length_436095_cov_65.793628_41,2.7.1.113,Metal:Mg(0.40),0.40,Probable,M-CSA,Metal:Mg: Ion: MG
2,NODE_1_length_436095_cov_65.793628_87,1.1.1.355,NADP+(0.50);NAD+(0.20);FAD(0.05);Fe-S cluster(...,0.50,Probable,EC-class-prior;KEGG;UniProt,NADP+: PDB CCD: NAP；NADPH/NADP+ 需根据反应方向选择氧化/还原...
3,NODE_1_length_436095_cov_65.793628_95,2.3.1.30,CoA(0.50),0.50,Probable,EC-class-prior;KEGG;UniProt,CoA: PDB CCD: COA；酰基-CoA 需根据底物改写
7,NODE_1_length_436095_cov_65.793628_144,3.1.1.5,CoA(0.70);Metal:Ca(0.30),0.70,High,M-CSA;UniProt,CoA: PDB CCD: COA；酰基-CoA 需根据底物改写 || Metal:Ca: ...
10,NODE_1_length_436095_cov_65.793628_206,2.3.1.87,CoA(0.90),0.90,High,EC-class-prior;KEGG;M-CSA;UniProt,CoA: PDB CCD: COA；酰基-CoA 需根据底物改写
11,NODE_1_length_436095_cov_65.793628_208,1.1.1.388,NAD+(0.50);NADP+(0.20);FAD(0.05);Fe-S cluster(...,0.50,Probable,EC-class-prior;KEGG;UniProt,NAD+: PDB CCD: NAD；NADH/NAD+ 需根据反应方向选择氧化/还原态 |...
16,NODE_1_length_436095_cov_65.793628_244,3.1.4.3,Metal:Zn(0.45);Metal:Ca(0.30);Metal:Cu(0.30);M...,0.45,Probable,KEGG;UniProt,Metal:Zn: Ion: ZN || Metal:Ca: Ion: CA || Meta...
17,NODE_1_length_436095_cov_65.793628_248,1.11.2.3;1.1.3.8,FAD(0.50);Heme(0.50);Metal:Ca(0.45);Metal:Fe(0...,0.50,Probable,EC-class-prior;KEGG;UniProt,FAD: PDB CCD: FAD || Heme: PDB CCD: HEM/HEC/HE...
19,NODE_1_length_436095_cov_65.793628_253,1.14.14.3,FMN(0.90);FAD(0.05);Fe-S cluster(0.05);Heme(0....,0.90,High,EC-class-prior;KEGG;M-CSA;UniProt,FMN: PDB CCD: FMN || FAD: PDB CCD: FAD || Fe-S...


In [4]:
# =========================================================
# Cell 7：构建 All-500 cofactor-aware modeling manifest
# 输入：EC_to_Cofactor_MultiEvidence_Mapping.xlsx
# 输出：主建模队列 / 二级建模队列 / apo-only 队列
# =========================================================

from google.colab import drive
drive.mount('/content/drive')

import re
import os
import pandas as pd
from pathlib import Path

# -----------------------------
# 0. 路径设置
# -----------------------------
mapping_xlsx = Path("/content/drive/MyDrive/Horizyn_Checkpoints/cofactor_mapping/EC_to_Cofactor_MultiEvidence_Mapping.xlsx")

# 你的结构文件目录，根据你的实际情况修改
# 如果你之前结构文件都在这个目录，就保持不变
pdb_dir = Path("/content/drive/MyDrive/Enzyme_Mining/best_conformations")

out_dir = Path("/content/drive/MyDrive/Horizyn_Checkpoints/cofactor_modeling_manifest")
out_dir.mkdir(parents=True, exist_ok=True)

assert mapping_xlsx.exists(), f"找不到 cofactor mapping 表: {mapping_xlsx}"

df = pd.read_excel(mapping_xlsx, sheet_name="enzyme_level")

print("输入酶数量:", len(df))
print("列名:")
print(df.columns.tolist())

# -----------------------------
# 1. 辅因子名称 → 建模 token / CCD hint 映射
# -----------------------------
COFACTOR_TO_MODEL_TOKEN = {
    "NAD+": {
        "model_token": "NAD",
        "type": "organic_cofactor",
        "note": "NAD+/NADH 状态需根据反应方向确认；可先用 NAD 作为 holo 建模 ligand"
    },
    "NADP+": {
        "model_token": "NAP",
        "type": "organic_cofactor",
        "note": "NADP+/NADPH 状态需根据反应方向确认；PDB CCD 常用 NAP"
    },
    "FAD": {
        "model_token": "FAD",
        "type": "organic_cofactor",
        "note": "Flavin adenine dinucleotide"
    },
    "FMN": {
        "model_token": "FMN",
        "type": "organic_cofactor",
        "note": "Flavin mononucleotide"
    },
    "PLP": {
        "model_token": "PLP",
        "type": "organic_cofactor",
        "note": "注意 PLP 可能与活性位点 Lys 形成 Schiff base"
    },
    "PQQ": {
        "model_token": "PQQ",
        "type": "organic_cofactor",
        "note": "Pyrroloquinoline quinone"
    },
    "SAM": {
        "model_token": "SAM",
        "type": "organic_cofactor",
        "note": "甲基转移反应也可考虑 SAH 作为产物态对照"
    },
    "CoA": {
        "model_token": "COA",
        "type": "organic_cofactor",
        "note": "如果反应涉及酰基转移，后续应考虑具体 acyl-CoA"
    },
    "Biotin": {
        "model_token": "BTN",
        "type": "organic_cofactor",
        "note": "注意生物素常共价连接 Lys，需要后续人工确认"
    },
    "TPP": {
        "model_token": "TPP",
        "type": "organic_cofactor",
        "note": "Thiamine pyrophosphate / thiamine diphosphate"
    },
    "Heme": {
        "model_token": "HEM",
        "type": "heme",
        "note": "需根据酶家族确认 HEM/HEC/HEA 类型"
    },
    "Cobalamin/B12": {
        "model_token": "B12",
        "type": "organic_cofactor",
        "note": "需确认 adenosylcobalamin 或 methylcobalamin"
    },
    "Tetrahydrofolate": {
        "model_token": "THF",
        "type": "organic_cofactor",
        "note": "THF 具体衍生物需根据反应确认"
    },
    "Lipoate": {
        "model_token": "LIP",
        "type": "organic_cofactor",
        "note": "常为 lipoamide arm，共价连接 Lys"
    },
    "Molybdopterin": {
        "model_token": "MPT",
        "type": "metal_complex_cofactor",
        "note": "需按具体家族确认 MPT/MGD/Moco 类型"
    },
    "Fe-S cluster": {
        "model_token": "SF4",
        "type": "metal_cluster",
        "note": "先按 [4Fe-4S] SF4 处理；需结合 Cys motif 判断 FES/F3S/SF4"
    },
    "Metal:Mg": {
        "model_token": "MG",
        "type": "metal_ion",
        "note": "Mg2+，后续需验证 Asp/Glu 配位几何"
    },
    "Metal:Zn": {
        "model_token": "ZN",
        "type": "metal_ion",
        "note": "Zn2+，后续需验证 His/Cys/Asp/Glu 配位几何"
    },
    "Metal:Fe": {
        "model_token": "FE",
        "type": "metal_ion",
        "note": "Fe2+/Fe3+ 价态需根据反应确认"
    },
    "Metal:Mn": {
        "model_token": "MN",
        "type": "metal_ion",
        "note": "Mn2+，后续需验证配位几何"
    },
    "Metal:Cu": {
        "model_token": "CU",
        "type": "metal_ion",
        "note": "Cu+/Cu2+ 价态需根据反应确认"
    },
    "Metal:Ca": {
        "model_token": "CA",
        "type": "metal_ion",
        "note": "Ca2+"
    },
    "Metal:Ni": {
        "model_token": "NI",
        "type": "metal_ion",
        "note": "Ni2+"
    },
    "Metal:Co": {
        "model_token": "CO",
        "type": "metal_ion",
        "note": "Co2+"
    },
    "Quinone": {
        "model_token": "UQ",
        "type": "organic_cofactor",
        "note": "需根据反应确认 ubiquinone / menaquinone / quinone 类型"
    },
}

# -----------------------------
# 2. 解析 Top_Cofactors / All_Cofactor_Candidates
# -----------------------------
def parse_cofactor_scores(s):
    """
    兼容：
    NADP+(0.50);NAD+(0.20);FAD(0.05)
    NADP+:0.50;NAD+:0.20
    """
    if pd.isna(s) or str(s).strip() == "":
        return []

    s = str(s)
    hits = []

    # 格式 1: Name(score)
    for name, score in re.findall(r"([^;()]+)\(([-+]?[0-9]*\.?[0-9]+)\)", s):
        name = name.strip()
        try:
            score = float(score)
        except:
            continue
        hits.append((name, score))

    # 格式 2: Name:score
    if not hits:
        for item in s.split(";"):
            item = item.strip()
            if not item or ":" not in item:
                continue

            name, score = item.rsplit(":", 1)
            name = name.strip()
            try:
                score = float(score)
            except:
                continue
            hits.append((name, score))

    # 去重，保留最高分
    best = {}
    for name, score in hits:
        if name not in best or score > best[name]:
            best[name] = score

    return sorted(best.items(), key=lambda x: x[1], reverse=True)

def classify_modeling_queue(row):
    cat = str(row.get("Cofactor_Category", ""))

    if cat in ["High-confidence cofactor-dependent", "Probable cofactor-dependent"]:
        return "Primary_holo"

    if cat == "Possible cofactor-dependent":
        return "Secondary_holo"

    if cat in ["Weak cofactor evidence", "No_public_evidence"]:
        return "Apo_only_control"

    return "Unclassified"

def find_pdb_path(row):
    """
    尝试根据 Enzyme_ID / Original_ID / CleanContact_Entry 找 PDB。
    这里不会强行报错，只记录是否找到。
    """
    possible_ids = []

    for c in ["Enzyme_ID", "Original_ID", "CleanContact_Entry"]:
        if c in row and pd.notna(row[c]):
            possible_ids.append(str(row[c]))

    if not pdb_dir.exists():
        return ""

    pdb_files = list(pdb_dir.glob("*.pdb"))

    # exact match
    for sid in possible_ids:
        for p in pdb_files:
            if p.stem == sid:
                return str(p)

    # contains match
    for sid in possible_ids:
        sid_lower = sid.lower()
        for p in pdb_files:
            if sid_lower in p.name.lower():
                return str(p)

    return ""

# -----------------------------
# 3. 生成建模任务
# -----------------------------
task_rows = []

for idx, row in df.iterrows():
    queue = classify_modeling_queue(row)

    enzyme_id = str(row.get("Enzyme_ID", row.get("CleanContact_Entry", f"enzyme_{idx}")))
    original_id = str(row.get("Original_ID", ""))
    clean_id = str(row.get("CleanContact_Entry", enzyme_id))

    top_cofactors = parse_cofactor_scores(row.get("Top_Cofactors", ""))
    all_cofactors = parse_cofactor_scores(row.get("All_Cofactor_Candidates", ""))

    if all_cofactors:
        cofactors = all_cofactors
    else:
        cofactors = top_cofactors

    pdb_path = find_pdb_path(row)

    # -------------------------
    # 队列规则
    # -------------------------
    if queue == "Primary_holo":
        # 主建模：保留 score >= 0.30 的辅因子
        # 如果所有都低于 0.30，则至少保留 top1
        selected = [(c, s) for c, s in cofactors if s >= 0.30]

        if not selected and cofactors:
            selected = [cofactors[0]]

        # NAD/NADP 特殊：如果二者都出现，即使一个是 0.20，也保留作为对照
        names = {c for c, s in cofactors}
        if "NAD+" in names or "NADP+" in names:
            for c, s in cofactors:
                if c in ["NAD+", "NADP+"] and (c, s) not in selected and s >= 0.20:
                    selected.append((c, s))

    elif queue == "Secondary_holo":
        # 二级建模：只保留 score >= 0.30 的辅因子
        selected = [(c, s) for c, s in cofactors if s >= 0.30]

        # 如果 Possible 但 top 分数很低，则不进入 holo，只记录为 Secondary_review
        if not selected and cofactors:
            selected = []

    else:
        selected = []

    # -------------------------
    # 生成任务
    # -------------------------
    if selected:
        for rank, (cof, score) in enumerate(selected, start=1):
            token_info = COFACTOR_TO_MODEL_TOKEN.get(cof, {
                "model_token": "",
                "type": "unknown",
                "note": "未在 token 字典中，需要人工确认"
            })

            task_rows.append({
                "Task_ID": f"{enzyme_id}__{cof.replace(':', '').replace('/', '_')}__rank{rank}",
                "Enzyme_ID": enzyme_id,
                "Original_ID": original_id,
                "CleanContact_Entry": clean_id,
                "Predicted_EC_4digit": row.get("Predicted_EC_4digit", ""),
                "Top1_EC": row.get("Top1_EC", ""),
                "CLEANContact_Distance": row.get("CLEANContact_Distance", ""),
                "Sequence_length": row.get("Sequence_length", ""),
                "Cofactor_Category": row.get("Cofactor_Category", ""),
                "Modeling_Queue": queue,
                "Cofactor": cof,
                "Cofactor_Score": score,
                "Cofactor_Confidence": row.get("Top_Cofactor_Confidence", ""),
                "Model_Token_or_CCD": token_info["model_token"],
                "Cofactor_Type": token_info["type"],
                "Modeling_Note": token_info["note"],
                "Evidence_Sources": row.get("Cofactor_Evidence_Sources", ""),
                "All_Cofactor_Candidates": row.get("All_Cofactor_Candidates", ""),
                "Input_PDB_Path": pdb_path,
                "PDB_Found": bool(pdb_path),
                "Recommended_Next_Tool": (
                    "DISCODE + Rossmann-toolbox" if cof in ["NAD+", "NADP+"] else
                    "Rossmann-toolbox / flavin motif / AlphaFill" if cof in ["FAD", "FMN"] else
                    "PLP Lys-motif + AlphaFill/BioLiP2" if cof == "PLP" else
                    "metal-site predictor + coordination check" if cof.startswith("Metal") or cof == "Fe-S cluster" else
                    "AlphaFill/BioLiP2 + holo cofolding"
                ),
            })
    else:
        task_rows.append({
            "Task_ID": f"{enzyme_id}__apo_control",
            "Enzyme_ID": enzyme_id,
            "Original_ID": original_id,
            "CleanContact_Entry": clean_id,
            "Predicted_EC_4digit": row.get("Predicted_EC_4digit", ""),
            "Top1_EC": row.get("Top1_EC", ""),
            "CLEANContact_Distance": row.get("CLEANContact_Distance", ""),
            "Sequence_length": row.get("Sequence_length", ""),
            "Cofactor_Category": row.get("Cofactor_Category", ""),
            "Modeling_Queue": "Apo_only_control" if queue == "Apo_only_control" else "Secondary_review_no_holo",
            "Cofactor": "",
            "Cofactor_Score": 0.0,
            "Cofactor_Confidence": row.get("Top_Cofactor_Confidence", ""),
            "Model_Token_or_CCD": "",
            "Cofactor_Type": "",
            "Modeling_Note": "No cofactor added at this stage",
            "Evidence_Sources": row.get("Cofactor_Evidence_Sources", ""),
            "All_Cofactor_Candidates": row.get("All_Cofactor_Candidates", ""),
            "Input_PDB_Path": pdb_path,
            "PDB_Found": bool(pdb_path),
            "Recommended_Next_Tool": "apo-only pocket analysis / structural homology review",
        })

df_tasks = pd.DataFrame(task_rows)

# -----------------------------
# 4. 拆分队列
# -----------------------------
df_primary = df_tasks[df_tasks["Modeling_Queue"].eq("Primary_holo")].copy()
df_secondary = df_tasks[df_tasks["Modeling_Queue"].eq("Secondary_holo")].copy()
df_apo = df_tasks[df_tasks["Modeling_Queue"].isin(["Apo_only_control", "Secondary_review_no_holo"])].copy()

# -----------------------------
# 5. 输出
# -----------------------------
all_out = out_dir / "01_All500_Modeling_Manifest.xlsx"
primary_out = out_dir / "02_Primary_Holo_Modeling_Tasks.xlsx"
secondary_out = out_dir / "03_Secondary_Holo_Modeling_Tasks.xlsx"
apo_out = out_dir / "04_Apo_Only_Control_Tasks.xlsx"

with pd.ExcelWriter(all_out, engine="openpyxl") as writer:
    df_tasks.to_excel(writer, index=False, sheet_name="all_modeling_tasks")
    df_primary.to_excel(writer, index=False, sheet_name="primary_holo")
    df_secondary.to_excel(writer, index=False, sheet_name="secondary_holo")
    df_apo.to_excel(writer, index=False, sheet_name="apo_or_review")

df_primary.to_excel(primary_out, index=False)
df_secondary.to_excel(secondary_out, index=False)
df_apo.to_excel(apo_out, index=False)

print("✅ All-500 modeling manifest 已生成")
print("总任务数:", len(df_tasks))
print("Primary holo tasks:", len(df_primary))
print("Secondary holo tasks:", len(df_secondary))
print("Apo/review tasks:", len(df_apo))

print("\n输出文件：")
print(all_out)
print(primary_out)
print(secondary_out)
print(apo_out)

print("\n按队列统计：")
print(df_tasks["Modeling_Queue"].value_counts(dropna=False))

print("\n按辅因子类型统计：")
print(df_tasks["Cofactor"].replace("", "Apo/no cofactor").value_counts().head(30))

print("\nPDB 匹配情况：")
print(df_tasks["PDB_Found"].value_counts(dropna=False))

display(df_tasks.head(20))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
输入酶数量: 500
列名:
['Enzyme_ID', 'Predicted_EC_4digit', 'CLEANContact_Distance', 'Top1_EC', 'Top1_Distance', 'Num_Predicted_ECs', 'Original_ID', 'CleanContact_Entry', 'Sequence_length', 'Parsed_ECs', 'Cofactor_Category', 'Top_Cofactors', 'Top_Cofactor_Score', 'Top_Cofactor_Confidence', 'All_Cofactor_Candidates', 'Cofactor_Evidence_Sources', 'Modeling_Hints']
✅ All-500 modeling manifest 已生成
总任务数: 1053
Primary holo tasks: 573
Secondary holo tasks: 335
Apo/review tasks: 145

输出文件：
/content/drive/MyDrive/Horizyn_Checkpoints/cofactor_modeling_manifest/01_All500_Modeling_Manifest.xlsx
/content/drive/MyDrive/Horizyn_Checkpoints/cofactor_modeling_manifest/02_Primary_Holo_Modeling_Tasks.xlsx
/content/drive/MyDrive/Horizyn_Checkpoints/cofactor_modeling_manifest/03_Secondary_Holo_Modeling_Tasks.xlsx
/content/drive/MyDrive/Horizyn_Checkpoints/cofactor_modeling_manifest/04_Ap

,Task_ID,Enzyme_ID,Original_ID,CleanContact_Entry,Predicted_EC_4digit,Top1_EC,CLEANContact_Distance,Sequence_length,Cofactor_Category,Modeling_Queue,...,Cofactor_Score,Cofactor_Confidence,Model_Token_or_CCD,Cofactor_Type,Modeling_Note,Evidence_Sources,All_Cofactor_Candidates,Input_PDB_Path,PDB_Found,Recommended_Next_Tool
0,NODE_1_length_436095_cov_65.793628_39__MetalMg...,NODE_1_length_436095_cov_65.793628_39,NODE_1_length_436095_cov_65.793628_39,NODE_1_length_436095_cov_65.793628_39,2.7.1.113;2.7.1.76,2.7.1.113,10.3859;10.4172,404,Probable cofactor-dependent,Primary_holo,...,0.4,Probable,MG,metal_ion,Mg2+，后续需验证 Asp/Glu 配位几何,M-CSA,Metal:Mg:0.40,/content/drive/MyDrive/Enzyme_Mining/best_conf...,True,metal-site predictor + coordination check
1,NODE_1_length_436095_cov_65.793628_41__MetalMg...,NODE_1_length_436095_cov_65.793628_41,NODE_1_length_436095_cov_65.793628_41,NODE_1_length_436095_cov_65.793628_41,2.7.1.113,2.7.1.113,8.8571,362,Probable cofactor-dependent,Primary_holo,...,0.4,Probable,MG,metal_ion,Mg2+，后续需验证 Asp/Glu 配位几何,M-CSA,Metal:Mg:0.40,/content/drive/MyDrive/Enzyme_Mining/best_conf...,True,metal-site predictor + coordination check
2,NODE_1_length_436095_cov_65.793628_87__NADP+__...,NODE_1_length_436095_cov_65.793628_87,NODE_1_length_436095_cov_65.793628_87,NODE_1_length_436095_cov_65.793628_87,1.1.1.355,1.1.1.355,9.0461,238,Probable cofactor-dependent,Primary_holo,...,0.5,Probable,NAP,organic_cofactor,NADP+/NADPH 状态需根据反应方向确认；PDB CCD 常用 NAP,EC-class-prior;KEGG;UniProt,NADP+:0.50;NAD+:0.20;FAD:0.05;Fe-S cluster:0.0...,/content/drive/MyDrive/Enzyme_Mining/best_conf...,True,DISCODE + Rossmann-toolbox
3,NODE_1_length_436095_cov_65.793628_87__NAD+__r...,NODE_1_length_436095_cov_65.793628_87,NODE_1_length_436095_cov_65.793628_87,NODE_1_length_436095_cov_65.793628_87,1.1.1.355,1.1.1.355,9.0461,238,Probable cofactor-dependent,Primary_holo,...,0.2,Probable,NAD,organic_cofactor,NAD+/NADH 状态需根据反应方向确认；可先用 NAD 作为 holo 建模 ligand,EC-class-prior;KEGG;UniProt,NADP+:0.50;NAD+:0.20;FAD:0.05;Fe-S cluster:0.0...,/content/drive/MyDrive/Enzyme_Mining/best_conf...,True,DISCODE + Rossmann-toolbox
4,NODE_1_length_436095_cov_65.793628_95__CoA__rank1,NODE_1_length_436095_cov_65.793628_95,NODE_1_length_436095_cov_65.793628_95,NODE_1_length_436095_cov_65.793628_95,2.3.1.30,2.3.1.30,7.6437,200,Probable cofactor-dependent,Primary_holo,...,0.5,Probable,COA,organic_cofactor,如果反应涉及酰基转移，后续应考虑具体 acyl-CoA,EC-class-prior;KEGG;UniProt,CoA:0.50,/content/drive/MyDrive/Enzyme_Mining/best_conf...,True,AlphaFill/BioLiP2 + holo cofolding
5,NODE_1_length_436095_cov_65.793628_115__MetalC...,NODE_1_length_436095_cov_65.793628_115,NODE_1_length_436095_cov_65.793628_115,NODE_1_length_436095_cov_65.793628_115,3.1.6.1,3.1.6.1,6.877,774,Possible cofactor-dependent,Secondary_holo,...,0.3,Possible,CA,metal_ion,Ca2+,KEGG;UniProt,Metal:Ca:0.30;Heme:0.15;Metal:Fe:0.15;Metal:Zn...,/content/drive/MyDrive/Enzyme_Mining/best_conf...,True,metal-site predictor + coordination check
6,NODE_1_length_436095_cov_65.793628_119__MetalM...,NODE_1_length_436095_cov_65.793628_119,NODE_1_length_436095_cov_65.793628_119,NODE_1_length_436095_cov_65.793628_119,3.1.21.2,3.1.21.2,7.9315,592,Possible cofactor-dependent,Secondary_holo,...,0.3,Possible,MN,metal_ion,Mn2+，后续需验证配位几何,UniProt,Metal:Mn:0.30;Metal:Zn:0.30,/content/drive/MyDrive/Enzyme_Mining/best_conf...,True,metal-site predictor + coordination check
7,NODE_1_length_436095_cov_65.793628_119__MetalZ...,NODE_1_length_436095_cov_65.793628_119,NODE_1_length_436095_cov_65.793628_119,NODE_1_length_436095_cov_65.793628_119,3.1.21.2,3.1.21.2,7.9315,592,Possible cofactor-dependent,Secondary_holo,...,0.3,Possible,ZN,metal_ion,Zn2+，后续需验证 His/Cys/Asp/Glu 配位几何,UniProt,Metal:Mn:0.30;Metal:Zn:0.30,/content/drive/MyDrive/Enzyme_Mining/best_conf...,True,metal-site predictor + coordination check
8,NODE_1_length_436095_cov_65.793628_142__apo_co...,NODE_1_length_436095_cov_65.793628_142,NODE_1_length_436095_cov_65.793628_142,NODE_1_length_436095_cov_65.793628_